# Multi-armed bandits

_Classical ML (beyond the cheat sheet)_

**Try arms to learn, but keep pulling the winner. Explore vs exploit.**

A multi-armed bandit is a row of slot machines (arms), each with an unknown payout.

     Every pull gives a reward and a clue about that arm.

---

This notebook is a practice scaffold for the **Multi-armed bandits** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — scikit-learn

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
true_p = np.array([0.3, 0.55, 0.7])                 # hidden win rates
K, T = len(true_p), 2000
sums = np.zeros(K); counts = np.zeros(K)

# seed: pull each arm once
for i in range(K):
    r = rng.random() < true_p[i]
    sums[i] += r; counts[i] += 1

for t in range(K, T):
    mean = sums / counts
    bonus = np.sqrt(2 * np.log(t) / counts)         # optimism bonus
    arm = int(np.argmax(mean + bonus))              # UCB1 choice
    r = rng.random() < true_p[arm]
    sums[arm] += r; counts[arm] += 1

best = int(np.argmax(true_p))
print("pulls per arm   :", counts.astype(int).tolist())
print("estimated means :", np.round(sums / counts, 3).tolist())
print("true best arm   :", best, " pulled %.1f%% of the time"
      % (100 * counts[best] / T))
total_reward = sums.sum()
print("avg reward/pull :", round(total_reward / T, 3),
      " (best possible %.3f)" % true_p[best])

## Visualize the data & results

_Across three ad creatives, which wins and does UCB beat random rotation?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

ads = ["Ad A: blue banner", "Ad B: video", "Ad C: carousel"]
true_ctr = np.array([0.06, 0.10, 0.16])  # real-looking click-through rates
K, T = len(true_ctr), 2000

def run(strategy, seed):
    rng = np.random.default_rng(seed)
    sums = np.zeros(K); counts = np.zeros(K); cum = []; total = 0.0
    for t in range(T):
        if t < K:
            arm = t
        elif strategy == "ucb":
            mean = sums / counts
            arm = int(np.argmax(mean + np.sqrt(2 * np.log(t) / counts)))
        else:
            arm = int(rng.integers(K))
        r = float(rng.random() < true_ctr[arm])
        sums[arm] += r; counts[arm] += 1; total += r; cum.append(total)
    return np.array(cum), counts.astype(int)

ucb_cum, ucb_counts = run("ucb", 0)
rnd_cum, _ = run("random", 1)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(np.arange(T), ucb_cum, c="#7ee787", label="UCB1")
ax[0].plot(np.arange(T), rnd_cum, c="#ff7b72", label="random")
ax[0].set_title("Cumulative clicks over 2000 impressions: UCB vs random rotation")
ax[0].set_xlabel("impression"); ax[0].set_ylabel("cumulative clicks"); ax[0].legend()
ax[1].bar(ads, ucb_counts, color=["#4ea1ff", "#4ea1ff", "#7ee787"])
ax[1].set_title("Impressions served per ad by UCB1")
ax[1].tick_params(axis="x", rotation=20)
plt.show()